# 📡 Notebook 1: Networking Fundamentals

Before diving into real-time update patterns, we need to understand the basics of how networks work. This knowledge will help you make better decisions when designing real-time systems.

## Learning Objectives

By the end of this notebook, you'll understand:
- The OSI model and why it matters for real-time systems
- TCP vs UDP trade-offs
- The HTTP request lifecycle
- Layer 4 vs Layer 7 load balancers

## 🧅 The OSI Model (Simplified)

Networks are built like an onion - in layers! Each layer provides services to the layer above it. As application developers, we mostly care about three layers:

```
┌─────────────────────────────────────────────────────────────┐
│  Layer 7: Application Layer                                 │
│  (HTTP, WebSocket, DNS, WebRTC)                             │
│  "What we usually work with"                                │
├─────────────────────────────────────────────────────────────┤
│  Layer 4: Transport Layer                                   │
│  (TCP, UDP)                                                 │
│  "Reliable delivery vs speed"                               │
├─────────────────────────────────────────────────────────────┤
│  Layer 3: Network Layer                                     │
│  (IP)                                                       │
│  "Getting packets from A to B"                              │
└─────────────────────────────────────────────────────────────┘
```

### Why does this matter for real-time systems?

Each layer adds some overhead and constraints. Understanding these helps us choose the right protocol for our needs.

## 🔄 TCP vs UDP

At Layer 4 (Transport), we have two main protocols:

### TCP (Transmission Control Protocol)
- **Connection-oriented**: Must establish connection first (handshake)
- **Reliable**: Guarantees delivery and order
- **Slower setup**: 3-way handshake adds latency

### UDP (User Datagram Protocol)
- **Connectionless**: Just send data, no setup needed
- **Unreliable**: Packets can be lost, duplicated, or reordered
- **Fast**: No handshake, minimal overhead

```
TCP: "Hey, are you there?" → "Yes, I'm here" → "Great, let's talk" → [data]
UDP: [data] → 🤷 (hope it arrives!)
```

In [ ]:
# Let's see TCP vs UDP in action!
import socket
import time

def demonstrate_tcp_handshake():
    """
    TCP requires a 3-way handshake before data transfer.
    This takes time but guarantees a reliable connection.
    """
    start = time.time()
    
    # Create a TCP socket
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(5)
    
    try:
        # This single line does:
        # 1. SYN → Server
        # 2. SYN-ACK ← Server  
        # 3. ACK → Server
        sock.connect(('httpbin.org', 80))
        elapsed = (time.time() - start) * 1000
        print(f"✅ TCP connection established in {elapsed:.2f}ms")
        print("   This included the 3-way handshake!")
    except Exception as e:
        print(f"❌ Connection failed: {e}")
    finally:
        sock.close()

demonstrate_tcp_handshake()

In [ ]:
# UDP is much simpler - no handshake needed
import socket

def demonstrate_udp_simplicity():
    """
    UDP doesn't require any connection setup -- and, more importantly, a
    send() *succeeds* whether or not anyone is listening. We prove both
    halves on loopback so this runs offline and deterministically.
    """
    message = b"Hello, UDP!"

    # 1. A real send to a real listener: no connect(), no handshake.
    receiver = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    receiver.bind(("127.0.0.1", 0))          # port 0 = let the OS pick one
    receiver.settimeout(2)
    port = receiver.getsockname()[1]

    sender = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    print("📤 Sending UDP packet (no handshake, no connect())...")
    sender.sendto(message, ("127.0.0.1", port))

    data, addr = receiver.recvfrom(1024)
    print(f"📥 Receiver got {data!r} from {addr[0]}:{addr[1]}")
    assert data == message, f"UDP payload was mangled: {data!r}"

    # 2. The same send to a port where NOTHING is listening. TCP would raise
    #    ConnectionRefusedError; UDP happily returns the byte count. This is
    #    the "unreliable" in "unreliable datagram" -- the sender cannot tell.
    closed_port = port + 1
    sent = sender.sendto(b"anybody there?", ("127.0.0.1", closed_port))
    print(f"\n📤 Sent {sent} bytes to port {closed_port} where nobody listens...")
    print("   ✅ sendto() returned normally. No error. No delivery either.")
    print("   👉 With TCP, connect() to a closed port raises ConnectionRefusedError.")

    receiver.close()
    sender.close()

demonstrate_udp_simplicity()

## 🌐 HTTP Request Lifecycle

When you type a URL in your browser, a lot happens under the hood:

```
┌─────────┐                                    ┌─────────┐
│ Browser │                                    │ Server  │
└────┬────┘                                    └────┬────┘
     │                                              │
     │  1. DNS Lookup (example.com → 93.184.216.34) │
     │─────────────────────────────────────────────>│
     │                                              │
     │  2. TCP Handshake                            │
     │──────────── SYN ────────────────────────────>│
     │<─────────── SYN-ACK ─────────────────────────│
     │──────────── ACK ────────────────────────────>│
     │                                              │
     │  3. HTTP Request                             │
     │──────────── GET /index.html ────────────────>│
     │                                              │
     │  4. HTTP Response                            │
     │<─────────── 200 OK + HTML ──────────────────│
     │                                              │
     │  5. TCP Teardown                             │
     │──────────── FIN ────────────────────────────>│
     │<─────────── ACK ─────────────────────────────│
     │<─────────── FIN ─────────────────────────────│
     │──────────── ACK ────────────────────────────>│
     │                                              │
```

### Key Insights for Real-time Systems:

1. **Every round trip adds latency** - Handshakes take time!
2. **Connections are stateful** - Both sides must track the connection
3. **HTTP is request-response** - Server can't push to client

In [ ]:
# Let's MEASURE the real cost of a fresh connection vs a reused one.
#
# Two things matter for honesty here:
#   * we report medians, not the mean of 3 samples -- one slow request over
#     the public internet would otherwise dominate the average;
#   * we print whatever the numbers say, including "inconclusive". A lab that
#     claims keep-alive won when its own measurement disagrees teaches you
#     to trust the story over the data.
import statistics
import time

import requests

def measure_http_overhead(samples: int = 7):
    url = "https://httpbin.org/get"   # public host => a real round trip

    def timed(get):
        t0 = time.perf_counter()
        get(url, timeout=10)
        return (time.perf_counter() - t0) * 1000

    try:
        # Test 1: a brand-new TCP + TLS connection for every request.
        print("🔄 Test 1: Fresh connections (new TCP + TLS handshake each time)")
        times_fresh = [timed(requests.get) for _ in range(samples)]
        for i, ms in enumerate(times_fresh, 1):
            print(f"   Request {i}: {ms:.2f}ms")

        # Test 2: one connection, reused. The FIRST request still pays for the
        # handshake, so we time it separately and exclude it from the median.
        print("\n🔗 Test 2: Reused connection (keep-alive)")
        session = requests.Session()
        first = timed(session.get)
        print(f"   Request 1: {first:.2f}ms  (still needs the handshake)")
        times_reuse = [timed(session.get) for _ in range(samples - 1)]
        for i, ms in enumerate(times_reuse, 2):
            print(f"   Request {i}: {ms:.2f}ms")
        session.close()
    except requests.exceptions.RequestException as exc:
        print(f"⏭️  Skipped: this cell needs internet access ({exc.__class__.__name__}).")
        print("   The lesson still holds: a fresh connection costs one TCP")
        print("   handshake (+2 more round trips for TLS) that a reused one skips.")
        return

    med_fresh = statistics.median(times_fresh)
    med_reuse = statistics.median(times_reuse)
    print(f"\n📊 Median fresh connection : {med_fresh:.2f}ms")
    print(f"📊 Median reused connection: {med_reuse:.2f}ms")

    saved = med_fresh - med_reuse
    if saved > 0:
        print(f"\n✅ Keep-alive saved {saved:.2f}ms per request "
              f"({saved / med_fresh:.0%}) -- that is the handshake you skipped.")
    else:
        print(f"\n⚠️  Inconclusive on this run ({saved:.2f}ms). Network jitter can")
        print("   swamp the handshake; re-run, or trust the mechanism over one sample.")

measure_http_overhead()

## ⚖️ Load Balancers: Layer 4 vs Layer 7

In production, you rarely have just one server. Load balancers distribute traffic across multiple servers. There are two main types:

### Layer 4 Load Balancer

Operates at the **transport layer** (TCP/UDP). Makes decisions based on:
- IP addresses
- Port numbers
- That's it!

```
┌────────┐      TCP Connection      ┌────────┐      TCP Connection      ┌────────┐
│ Client │◄────────────────────────►│ L4 LB  │◄────────────────────────►│ Server │
└────────┘                          └────────┘                          └────────┘

The L4 LB just forwards TCP packets. It's like a transparent pipe.
The client effectively has a direct TCP connection to one server.
```

**Pros:** Fast, efficient, maintains persistent connections
**Cons:** Can't make smart routing decisions based on request content

### Layer 7 Load Balancer

Operates at the **application layer** (HTTP). Can inspect:
- URL paths
- Headers
- Cookies
- Request body

```
┌────────┐    TCP #1    ┌────────┐    TCP #2    ┌────────┐
│ Client │◄────────────►│ L7 LB  │◄────────────►│ Server │
└────────┘              └────────┘              └────────┘

The L7 LB terminates the client connection and creates a new one to the server.
It can route different requests to different servers!
```

**Pros:** Smart routing, can route `/api` to one server and `/static` to another
**Cons:** More CPU overhead, breaks persistent connections

## 🎯 Why This Matters for Real-time Systems

| Protocol | Load Balancer | Works Well? | Why |
|----------|--------------|-------------|-----|
| Simple Polling | L4 or L7 | ✅ | Just HTTP requests |
| Long Polling | L4 or L7 | ✅ | Still just HTTP |
| SSE | L7 (must support streaming) | ⚠️ | Need streaming support |
| WebSocket | L4 preferred | ✅ | Needs persistent TCP |
| WebSocket | L7 (with WS support) | ⚠️ | Not all L7 LBs support it |

### Key Takeaways:

1. **L4 Load Balancers** are better for WebSockets because they maintain the TCP connection
2. **L7 Load Balancers** work better for HTTP-based solutions like polling
3. Many modern L7 LBs (AWS ALB, nginx) now support WebSocket "upgrade"

In [ ]:
# Let's visualize the concept of connection persistence

def visualize_lb_behavior():
    """
    This demonstrates conceptually how L4 vs L7 LBs handle connections.
    """
    
    print("📊 Layer 4 Load Balancer Behavior")
    print("="*50)
    print("")
    print("Client connects → LB picks Server A")
    print("Request 1 → Server A  ✅")
    print("Request 2 → Server A  ✅ (same TCP connection)")
    print("Request 3 → Server A  ✅ (same TCP connection)")
    print("")
    print("👍 All requests go to the same server!")
    print("   Great for WebSockets and stateful connections.")
    print("")
    
    print("📊 Layer 7 Load Balancer Behavior")
    print("="*50)
    print("")
    print("Request 1: GET /api/users")
    print("  → LB inspects path → Routes to API Server")
    print("")
    print("Request 2: GET /static/logo.png")
    print("  → LB inspects path → Routes to Static Server")
    print("")
    print("Request 3: POST /api/orders")
    print("  → LB inspects path → Routes to API Server")
    print("")
    print("👍 Smart routing based on content!")
    print("⚠️  Each request might go to a different backend server.")

visualize_lb_behavior()

## 🧪 Quick Quiz

Test your understanding! Try to answer before running the cell.

1. **Which protocol would you use for a video call?**
   - A) TCP (reliable)
   - B) UDP (fast)

2. **You're building a chat app with WebSockets. Which load balancer type is better?**
   - A) Layer 4
   - B) Layer 7

3. **How many round trips does a TCP handshake take?**
   - A) 1
   - B) 1.5 (3 messages)
   - C) 3

In [ ]:
# Run this cell to see the answers!

def show_quiz_answers():
    print("📝 Quiz Answers")
    print("="*50)
    print("")
    print("1. B) UDP - Video calls prioritize speed over reliability.")
    print("   A few dropped frames is better than frozen video!")
    print("")
    print("2. A) Layer 4 - WebSockets need persistent TCP connections.")
    print("   L4 LBs maintain the connection to the same server.")
    print("")
    print("3. B) 1.5 round trips (3 messages: SYN, SYN-ACK, ACK)")
    print("   The client can start sending data after the 3rd message.")

show_quiz_answers()

## 🔒 In Production: TLS, `https://`, `wss://`

Every protocol you'll meet in this lab has an **encrypted twin** used in
production. On localhost we use the plain versions so you can see what's
happening with tools like `curl` and your browser's dev tools, but real
deployments almost always run over TLS.

```
Plain (dev only)        Encrypted (production)
────────────────        ──────────────────────
http://                 https://
ws://                   wss://       (WebSocket over TLS)
(SSE uses https://)     (SSE over TLS is just https://)
```

### Why it matters

1. **Corporate proxies and browsers block plain `ws://`** on `https://` pages.
   If your site is `https://app.example.com`, WebSockets **must** use `wss://`.
2. **TLS adds latency only during the handshake** — ~1 extra round trip (or 0
   with TLS 1.3 resumption). Once connected, encryption overhead is tiny.
3. **Layer-7 load balancers need to speak your protocol** (and often terminate
   TLS): AWS ALB, nginx, and Cloudflare all need the "WebSocket" or "SSE"
   feature explicitly enabled.

> 💡 Rule of thumb: **write the code with `ws://` / `http://` locally, then
> flip to `wss://` / `https://` in your config for deploy.** The application
> code is identical.


## 📚 Summary

### What We Learned:

1. **OSI Layers** - Networks are layered; we mostly care about L3 (IP), L4 (TCP/UDP), L7 (HTTP)

2. **TCP vs UDP**:
   - TCP: Reliable but slow setup (handshake)
   - UDP: Fast but unreliable (no guarantees)

3. **HTTP Lifecycle**: DNS → TCP Handshake → Request → Response → Teardown

4. **Load Balancers**:
   - L4: Fast, maintains connections, can't inspect content
   - L7: Smart routing, but breaks persistent connections

### Next Up: Simple Polling

In the next notebook, we'll build our first real-time update system using simple polling - the most straightforward approach!